# 27j MoE-Ensemble Teacher Distillation

- created_utc: 2026-05-11T17:02:11+00:00
- target: scene_daynight_total mAP50 >= 0.600
- workspace: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27j_moe_ensemble_teacher_distill_r1`
- start/student anchor: `25a_r1_repair` (27i best single checkpoint, mAP50=0.464)
- pseudo teacher: 8-checkpoint model-level MoE ensemble

## What We Learned Before This

- Ordinary DQA aggregation and short MoE probes saturate around mAP50=0.462.
- 27h model-level/test-time MoE reached mAP50=0.464 / mAP50:95=0.262, the best observed output predictor.
- 27i found no hidden stronger old checkpoint; `25a_r1_repair` is the best single teacher at mAP50=0.464 / mAP50:95=0.261.

## Paper Basis

- [Domain-Specialized Object Detection via Model-Level Mixtures of Experts](https://arxiv.org/abs/2604.18256): BDD100K object detectors can benefit from model-level expert fusion and domain-specialized experts.
- [HI-MoE](https://arxiv.org/abs/2604.04908): detection MoE should route at scene and instance/object granularity rather than only image-level routing.
- [STEP-DETR](https://openaccess.thecvf.com/content/ICCV2025/papers/Shehzadi_STEP-DETR_Advancing_DETR-based_Semi-Supervised_Object_Detection_with_Super_Teacher_and_ICCV_2025_paper.pdf): SSOD improves when a stronger teacher explicitly supplies higher-quality pseudo labels and reduces confidence bias.

## Hypothesis

Use the single best checkpoint as a stable student anchor, but generate pseudo labels from a model-level MoE ensemble containing global repair experts and night client specialists. Train only MoE-head/router slots for one round so the model learns the ensemble's domain signal without moving the detector body destructively.


In [1]:
import json
import subprocess
from pathlib import Path

REPO_ROOT = Path('/app/Object_Detection')
WORKSPACE = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27j_moe_ensemble_teacher_distill_r1')
LOG_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27j_moe_ensemble_teacher_distill_r1/logs/27j_moe_ensemble_teacher_distill_r1_train.log')
CMD = ['/opt/venv/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_18_client_balanced_single_injection_dqamox.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27j_moe_ensemble_teacher_distill_r1', '--repair-baseline-rounds', '0', '--source-workspace', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup', '--source-repair-baseline-rounds', '30', '--target-map50', '0.60', '--skip-warmup-training', '--warmup-checkpoint', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/25_paper_round_until_target/25a_fedmox50_sto20_30_top1/checkpoints/latent_dqamox_p1_round001_server_repair.pt', '--pseudo-teacher-checkpoints', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup/checkpoints/round000_latent_dqamox_warmup.pt,/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/25_paper_round_until_target/25a_fedmox50_sto20_30_top1/checkpoints/latent_dqamox_p1_round001_server_repair.pt,/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27d_probe_teacher_residual_mixpl_r2/checkpoints/latent_dqamox_p1_round002_server_repair.pt,/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27e_probe_clean_day_expert_anchor_r2/checkpoints/latent_dqamox_p1_round002_server_repair.pt,/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27g_probe_moe_head_only_router_r1/checkpoints/latent_dqamox_p1_round001_server_repair.pt,/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27g_probe_moe_head_only_router_r1/checkpoints/latent_dqamox_p1_round001_client1_highway_night.pt,/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27g_probe_moe_head_only_router_r1/checkpoints/latent_dqamox_p1_round001_client3_citystreet_night.pt,/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27g_probe_moe_head_only_router_r1/checkpoints/latent_dqamox_p1_round001_client5_residential_night.pt', '--num-experts', '4', '--top-k', '2', '--router-temperature', '0.90', '--router-balance-weight', '0.040', '--router-entropy-weight', '0.0002', '--dqa-client-balance-stats', '--dqa-client-balance-target', 'median', '--dqa-client-balance-max-scale', '4.0', '--load-bias-strength', '0.22', '--batch-size', '80', '--workers', '8', '--gpus', '2', '--client-limit', '800', '--client-sampling-ratio', '1.000', '--client-sampling-seed', '270613', '--phase1-rounds', '1', '--phase2-rounds', '0', '--phase1-train-scope', 'moe_head', '--phase1-repair-train-scope', 'moe_head', '--phase1-client-epochs', '1', '--phase1-client-lr', '0.00032', '--phase1-source-repeat', '4', '--phase1-pseudo-repeat', '2', '--phase1-loss-box', '0.00008', '--server-repair-epochs', '1', '--server-repair-lr', '0.00020', '--server-repair-loss-box', '0.0005', '--dqa-temperature', '0.80', '--dqa-uniform-mix', '0.10', '--dqa-classwise-blend', '0.28', '--dqa-stability-lambda', '0.55', '--dqa-server-anchor', '0.70', '--dqa-min-server-alpha', '0.64', '--dqa-residual-blend', '0.08', '--curriculum-start-round', '2', '--expert-keep-fraction', '0.90', '--expert-max-class-fraction', '0.36', '--actual-max-class-fraction', '0.46', '--min-score', '0.14', '--min-stability', '0.46', '--max-boxes-per-image', '14', '--imgsz', '640', '--conf-thres', '0.20', '--nms-iou-thres', '0.65', '--match-iou', '0.58', '--min-views', '2', '--max-images-per-client', '0', '--master-port', '39400', '--evaluate', '--classwise', '--no-eval-plots', '--force', '--force-pseudo', '--notify-start', '--notify-end']

WORKSPACE.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats").mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats" / "27j_notebook_command.json").write_text(
    json.dumps({"command": CMD}, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(" ".join(CMD))
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.run(CMD, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, check=False)
print("returncode", proc.returncode)
print("log", LOG_PATH)
if proc.returncode != 0:
    raise SystemExit(proc.returncode)


/opt/venv/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_18_client_balanced_single_injection_dqamox.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27j_moe_ensemble_teacher_distill_r1 --repair-baseline-rounds 0 --source-workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup --source-repair-baseline-rounds 30 --target-map50 0.60 --skip-warmup-training --warmup-checkpoint /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/25_paper_round_until_target/25a_fedmox50_sto20_30_top1/checkpoints/latent_dqamox_p1_round001_server_repair.pt --pseudo-teacher-checkpoints /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_late

returncode 0
log /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27j_moe_ensemble_teacher_distill_r1/logs/27j_moe_ensemble_teacher_distill_r1_train.log


In [2]:
import csv
import math
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path('/app/Object_Detection')
WORKSPACE = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27j_moe_ensemble_teacher_distill_r1')
METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27j_moe_ensemble_teacher_distill_r1/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
SUMMARY_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/reports/27_research_loop_mAP_summary.csv')
NOTEBOOK_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/notebooks/research_loop_until_060/004_27j_moe_ensemble_teacher_distill_r1.ipynb')
LOG_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27j_moe_ensemble_teacher_distill_r1/logs/27j_moe_ensemble_teacher_distill_r1_train.log')
TRIAL = "27j_moe_ensemble_teacher_distill_r1"
RATIONALE = (
    "27h showed model-level MoE is the best existing predictor, while 27i found 25a_r1_repair "
    "is the best single teacher. 27j uses the single teacher as the student anchor and an "
    "8-checkpoint model-level MoE ensemble only for pseudo-label generation, distilling that "
    "scene/night specialist signal into MoE-head/router slots."
)

rows = list(csv.DictReader(METRICS_PATH.open(encoding="utf-8"))) if METRICS_PATH.exists() else []
for row in rows:
    print(row)

def f(raw):
    try:
        value = float(raw or "nan")
    except ValueError:
        return None
    return value if math.isfinite(value) else None

warm = next((row for row in rows if row.get("kind") == "warmup"), {})
best_row = max(rows, key=lambda row: f(row.get("map50")) or -1.0) if rows else {}
summary_row = {
    "trial": TRIAL,
    "status": "target_reached" if (f(best_row.get("map50")) or 0.0) >= 0.60 else "completed",
    "best_map50": best_row.get("map50", ""),
    "best_map50_95": best_row.get("map50_95", ""),
    "warmup_map50": warm.get("map50", ""),
    "repair_map50": "",
    "dqa_aggregate_map50": next((row.get("map50", "") for row in rows if row.get("kind") == "aggregate"), ""),
    "dqa_repair_map50": rows[-1].get("map50", "") if rows else "",
    "workspace": str(WORKSPACE),
    "notebook": str(NOTEBOOK_PATH),
    "log": str(LOG_PATH),
    "finished_utc": datetime.now(timezone.utc).isoformat(),
    "rationale": RATIONALE,
}
fields = [
    "trial", "status", "best_map50", "best_map50_95", "warmup_map50", "repair_map50",
    "dqa_aggregate_map50", "dqa_repair_map50", "workspace", "notebook", "log",
    "finished_utc", "rationale",
]
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
existing = list(csv.DictReader(SUMMARY_PATH.open(encoding="utf-8"))) if SUMMARY_PATH.exists() else []
existing = [row for row in existing if row.get("trial") != TRIAL]
existing.append(summary_row)
with SUMMARY_PATH.open("w", encoding="utf-8", newline="") as fobj:
    writer = csv.DictWriter(fobj, fieldnames=fields)
    writer.writeheader()
    writer.writerows(existing)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
try:
    from notebook_notify import notify_discord

    msg = "\n".join([
        "27j finished: MoE-ensemble pseudo-teacher distillation",
        f"best_mAP50={summary_row['best_map50']} / mAP50:95={summary_row['best_map50_95']}",
        f"warmup_mAP50={summary_row['warmup_map50']}",
        f"dqa_aggregate_mAP50={summary_row['dqa_aggregate_map50']}",
        f"dqa_repair_mAP50={summary_row['dqa_repair_map50']}",
        f"workspace={WORKSPACE}",
    ])
    print(notify_discord(msg, title="DQA-MoX 27j result", fail_silently=True))
except Exception as exc:
    print("Discord notification skipped:", exc)


{'checkpoint_label': 'warmup_global', 'condition': 'warmup', 'kind': 'warmup', 'phase': '', 'round': '', 'precision': '0.719', 'recall': '0.414', 'map50': '0.464000', 'map50_95': '0.261000', 'gain_vs_warmup_map50_95': '0.000000', 'delta_vs_server_repair_map50_95': '0.060000', 'worst_split': 'highway_night', 'worst_split_map50_95': '0.174000', 'day_avg_map50_95': '0.284000', 'night_avg_map50_95': '0.201000', 'day_night_gap_map50_95': '0.083000'}
{'checkpoint_label': 'warmup_server_repair_final', 'condition': 'warmup + server repair', 'kind': 'server_repair', 'phase': '0', 'round': '30', 'precision': '0.662', 'recall': '0.366', 'map50': '0.378000', 'map50_95': '0.201000', 'gain_vs_warmup_map50_95': '-0.060000', 'delta_vs_server_repair_map50_95': '0.000000', 'worst_split': 'highway_night', 'worst_split_map50_95': '0.134000', 'day_avg_map50_95': '0.226000', 'night_avg_map50_95': '0.148667', 'day_night_gap_map50_95': '0.077333'}
{'checkpoint_label': 'latent_dqamox_final_aggregate', 'conditi

DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)
